In [ ]:
# ---- Startup cell: run this first, every time ----
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE = "/content/drive/MyDrive/legal-retrieval-eval"
for folder in ["data/raw", "data/processed", "src", "eval", "notebooks", "paper"]:
    os.makedirs(f"{BASE}/{folder}", exist_ok=True)

# so you can `import metrics` (and later, your other code)
sys.path.append(f"{BASE}/eval")
sys.path.append(f"{BASE}/src")

# libraries (reinstalled each session; takes ~1 minute)
!pip install -q rank_bm25 sentence-transformers transformers nltk pandas tqdm datasets

# Hugging Face login (only matters once the dataset is approved)
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("HF login skipped:", e)

print("Ready. Base folder:", os.listdir(BASE))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready. Base folder: ['data', 'src', 'eval', 'notebooks', 'paper']


In [ ]:
print("hello legal retrieval")

hello legal retrieval


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
base = "/content/drive/MyDrive/legal-retrieval-eval"
for folder in ["data/raw", "data/processed", "src", "eval", "notebooks", "paper"]:
    os.makedirs(f"{base}/{folder}", exist_ok=True)
print(os.listdir(base))

['data', 'src', 'eval', 'notebooks', 'paper']


In [ ]:
!pip install -q rank_bm25 sentence-transformers transformers nltk pandas tqdm

In [ ]:
import rank_bm25, sentence_transformers, pandas
print("all libraries imported fine")

all libraries imported fine


In [ ]:
!pip install -q datasets
from datasets import get_dataset_config_names
print(get_dataset_config_names("Exploration-Lab/IL-TUR"))

README.md:   0%|          | 0.00/37.9k [00:00<?, ?B/s]

DatasetNotFoundError: Dataset 'Exploration-Lab/IL-TUR' is a gated dataset on the Hub. You must be authenticated to access it.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

In [ ]:
def precision_recall_f1_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]
    hits = len(set(top_k) & set(relevant))
    precision = hits / k
    recall = hits / len(relevant)
    f1 = 2 * precision * recall / (precision + recall) if hits > 0 else 0.0
    return precision, recall, f1

# toy example: the system's ranking (best first) and the truly cited cases
retrieved = ["c3", "c7", "c1", "c9", "c5"]
relevant  = ["c1", "c5", "c8"]

for k in [1, 3, 5]:
    print(k, precision_recall_f1_at_k(retrieved, relevant, k))

1 (0.0, 0.0, 0.0)
3 (0.3333333333333333, 0.3333333333333333, 0.3333333333333333)
5 (0.4, 0.6666666666666666, 0.5)


In [ ]:
def reciprocal_rank(retrieved, relevant):
    for i, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            return 1 / i
    return 0.0

def average_precision(retrieved, relevant):
    hits = 0
    total = 0.0
    for i, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            hits += 1
            total += hits / i
    return total / len(relevant)

# query 1 (same toy example as before)
retrieved1 = ["c3", "c7", "c1", "c9", "c5"]
relevant1  = ["c1", "c5", "c8"]
# query 2
retrieved2 = ["a", "b", "c"]
relevant2  = ["a"]

print("RR  q1:", reciprocal_rank(retrieved1, relevant1))
print("AP  q1:", average_precision(retrieved1, relevant1))
print("RR  q2:", reciprocal_rank(retrieved2, relevant2))
print("AP  q2:", average_precision(retrieved2, relevant2))

print("MRR:", (reciprocal_rank(retrieved1, relevant1) + reciprocal_rank(retrieved2, relevant2)) / 2)
print("MAP:", (average_precision(retrieved1, relevant1) + average_precision(retrieved2, relevant2)) / 2)

RR  q1: 0.3333333333333333
AP  q1: 0.24444444444444446
RR  q2: 1.0
AP  q2: 1.0
MRR: 0.6666666666666666
MAP: 0.6222222222222222


In [ ]:
import math

def ndcg_at_k(retrieved, relevant, k):
    dcg = sum(1 / math.log2(i + 1)
              for i, doc in enumerate(retrieved[:k], start=1)
              if doc in relevant)
    ideal_hits = min(k, len(relevant))
    idcg = sum(1 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg

retrieved1 = ["c3", "c7", "c1", "c9", "c5"]
relevant1  = ["c1", "c5", "c8"]
retrieved2 = ["a", "b", "c"]
relevant2  = ["a"]

n1 = ndcg_at_k(retrieved1, relevant1, 5)
n2 = ndcg_at_k(retrieved2, relevant2, 5)
print("nDCG@5 q1:", n1)
print("nDCG@5 q2:", n2)
print("mean nDCG@5:", (n1 + n2) / 2)

nDCG@5 q1: 0.41618115554873086
nDCG@5 q2: 1.0
mean nDCG@5: 0.7080905777743655


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
base = "/content/drive/MyDrive/legal-retrieval-eval"
for folder in ["data/raw", "data/processed", "src", "eval", "notebooks", "paper"]:
    os.makedirs(f"{base}/{folder}", exist_ok=True)
print(os.listdir(base))

Mounted at /content/drive
['data', 'src', 'eval', 'notebooks', 'paper']


In [ ]:
%%writefile /content/drive/MyDrive/legal-retrieval-eval/eval/metrics.py
import math

def precision_recall_f1_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]
    hits = len(set(top_k) & set(relevant))
    precision = hits / k
    recall = hits / len(relevant)
    f1 = 2 * precision * recall / (precision + recall) if hits > 0 else 0.0
    return precision, recall, f1

def reciprocal_rank(retrieved, relevant):
    for i, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            return 1 / i
    return 0.0

def average_precision(retrieved, relevant):
    hits = 0
    total = 0.0
    for i, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            hits += 1
            total += hits / i
    return total / len(relevant)

def ndcg_at_k(retrieved, relevant, k):
    dcg = sum(1 / math.log2(i + 1)
              for i, doc in enumerate(retrieved[:k], start=1)
              if doc in relevant)
    ideal_hits = min(k, len(relevant))
    idcg = sum(1 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg

Writing /content/drive/MyDrive/legal-retrieval-eval/eval/metrics.py


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/legal-retrieval-eval/eval")
import metrics

print(metrics.reciprocal_rank(["c3", "c7", "c1"], ["c1"]))   # expect 0.3333
print(metrics.ndcg_at_k(["a", "b", "c"], ["a"], 5))          # expect 1.0

0.3333333333333333
1.0


In [ ]:

%%writefile /content/drive/MyDrive/legal-retrieval-eval/eval/metrics.py

import math

def precision_recall_f1_at_k(retrieved, relevant, k):
    if k <= 0:
        raise ValueError("k must be greater than 0")

    if not relevant:
        return 0.0, 0.0, 0.0

    top_k = retrieved[:k]
    hits = len(set(top_k) & set(relevant))

    precision = hits / k
    recall = hits / len(set(relevant))

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0.0
    )

    return precision, recall, f1


def reciprocal_rank(retrieved, relevant):
    relevant = set(relevant)

    if not relevant:
        return 0.0

    for i, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            return 1 / i

    return 0.0


def average_precision(retrieved, relevant):
    relevant = set(relevant)

    if not relevant:
        return 0.0

    hits = 0
    total = 0.0
    seen = set()

    for i, doc in enumerate(retrieved, start=1):
        if doc in relevant and doc not in seen:
            hits += 1
            total += hits / i
            seen.add(doc)

    return total / len(relevant)


def ndcg_at_k(retrieved, relevant, k):
    if k <= 0:
        raise ValueError("k must be greater than 0")

    relevant = set(relevant)

    if not relevant:
        return 0.0

    dcg = 0.0
    seen = set()

    for i, doc in enumerate(retrieved[:k], start=1):
        if doc in relevant and doc not in seen:
            dcg += 1 / math.log2(i + 1)
            seen.add(doc)

    ideal_hits = min(k, len(relevant))

    idcg = sum(
        1 / math.log2(i + 1)
        for i in range(1, ideal_hits + 1)
    )

    return dcg / idcg


Overwriting /content/drive/MyDrive/legal-retrieval-eval/eval/metrics.py


In [ ]:

import sys
import importlib

sys.path.append("/content/drive/MyDrive/legal-retrieval-eval/eval")

import metrics
importlib.reload(metrics)

retrieved = ["c3", "c7", "c1", "c9", "c5"]
relevant = ["c1", "c5", "c8"]

print("Precision, Recall, F1:",
      metrics.precision_recall_f1_at_k(retrieved, relevant, 5))

print("MRR:", metrics.reciprocal_rank(retrieved, relevant))
print("MAP:", metrics.average_precision(retrieved, relevant))
print("nDCG@5:", metrics.ndcg_at_k(retrieved, relevant, 5))

print("Empty relevance test:",
      metrics.reciprocal_rank(["c1"], []))


Precision, Recall, F1: (0.4, 0.6666666666666666, 0.5)
MRR: 0.3333333333333333
MAP: 0.24444444444444446
nDCG@5: 0.41618115554873086
Empty relevance test: 0.0


In [ ]:

import os

project_path = "/content/drive/MyDrive/legal-retrieval-eval"

for folder in ["data/raw", "data/processed", "src", "eval", "notebooks", "paper"]:
    path = os.path.join(project_path, folder)
    print(folder, "→", "Exists" if os.path.exists(path) else "Missing")


data/raw → Exists
data/processed → Exists
src → Exists
eval → Exists
notebooks → Exists
paper → Exists


In [ ]:

import os

raw_path = "/content/drive/MyDrive/legal-retrieval-eval/data/raw"

print("Checking AILA dataset upload...\n")

if os.path.exists(raw_path):
    files = os.listdir(raw_path)

    for file in files:
        full_path = os.path.join(raw_path, file)

        if os.path.isfile(full_path):
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            print(f"{file} — {size_mb:.2f} MB")
        else:
            print(f"{file} — Folder")
else:
    print("The data/raw folder was not found.")


Checking AILA dataset upload...

AILA_2019_dataset.zip — 21.06 MB


In [ ]:

import zipfile
import os

zip_path = "/content/drive/MyDrive/legal-retrieval-eval/data/raw/AILA_2019_dataset.zip"

extract_path = "/content/drive/MyDrive/legal-retrieval-eval/data/raw/AILA_2019_dataset"

print("Extracting AILA dataset...")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed!")

print("\nExtracted items:")
for item in os.listdir(extract_path):
    print("-", item)


Extracting AILA dataset...
Extraction completed!

Extracted items:
- AILA_2019_dataset
- __MACOSX


In [ ]:

import os

base_path = "/content/drive/MyDrive/legal-retrieval-eval/data/raw/AILA_2019_dataset"

print("Checking extracted dataset structure...\n")

for root, dirs, files in os.walk(base_path):
    dirs[:] = [d for d in dirs if d != "__MACOSX"]

    level = root.replace(base_path, "").count(os.sep)
    if level > 2:
        dirs[:] = []
        continue

    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:5]:
        if not file.startswith("._"):
            print(f"{indent}    {file}")

    if len(files) > 5:
        print(f"{indent}    ... and {len(files)-5} more files")


Checking extracted dataset structure...

AILA_2019_dataset/
    AILA_2019_dataset/
        AILA_2019_Overview_Paper.pdf
        Query_doc.txt
        relevance_judgments_statutes.txt
        relevance_judgments_priorcases.txt
        README.txt
        Object_statutes/
            S168.txt
            S154.txt
            S140.txt
            S197.txt
            S183.txt
            ... and 192 more files
        Object_casedocs/
            C1763.txt
            C455.txt
            C333.txt
            C1005.txt
            C1011.txt
            ... and 2909 more files


In [ ]:

import os

dataset_path = "/content/drive/MyDrive/legal-retrieval-eval/data/raw/AILA_2019_dataset/AILA_2019_dataset"

# Count case documents
case_folder = os.path.join(dataset_path, "Object_casedocs")
case_files = [f for f in os.listdir(case_folder) if f.endswith(".txt")]

# Count statute documents
statute_folder = os.path.join(dataset_path, "Object_statutes")
statute_files = [f for f in os.listdir(statute_folder) if f.endswith(".txt")]

# Read queries
query_path = os.path.join(dataset_path, "Query_doc.txt")

with open(query_path, "r", encoding="utf-8") as f:
    queries = [line.strip() for line in f if line.strip()]

# Read relevance judgments
qrels_path = os.path.join(dataset_path, "relevance_judgments_priorcases.txt")

with open(qrels_path, "r", encoding="utf-8") as f:
    qrels = [line.strip() for line in f if line.strip()]

print("===== AILA DATASET VERIFICATION =====")
print("Case documents:", len(case_files))
print("Statute documents:", len(statute_files))
print("Queries:", len(queries))
print("Prior-case relevance judgments:", len(qrels))

print("\n===== FIRST QUERY =====")
print(queries[0])

print("\n===== FIRST CASE DOCUMENT =====")
first_case = os.path.join(case_folder, "C1.txt")

with open(first_case, "r", encoding="utf-8", errors="replace") as f:
    print(f.read()[:1000])

print("\n===== FIRST RELEVANCE JUDGMENT =====")
print(qrels[0])


===== AILA DATASET VERIFICATION =====
Case documents: 2914
Statute documents: 197
Queries: 50
Prior-case relevance judgments: 145700

===== FIRST QUERY =====
AILA_Q1||The appellant on February 9, 1961 was appointed as an Officer in Grade III in the respondent Bank ( for short 'the Bank'). He was promoted on April 1, 1968 to the Grade officer in the Foreign Exchange Department in the Head Office of the Bank. Sometime in 1964, MCH Society ( for short 'the Society') was formed of which the appellant was one of the chief promoters and thereafter its Secretary. The object of the Society was to construct residential premises for the employees of the Bank and its other members. It appears that the complaint was received in respect of the affairs of the Society relating to misappropriation of the funds of the Society and consequently, in exercise of the powers under Section S of Act A1, the Registrar on April 23, 1969 instituted an inquiry thereof. P1 was appointed the Registrar's nominee who 

In [ ]:

import pandas as pd
import os

# Load relevance judgments
qrels_path = os.path.join(
    dataset_path,
    "relevance_judgments_priorcases.txt"
)

qrels_df = pd.read_csv(
    qrels_path,
    sep=r"\s+",
    header=None,
    names=["query_id", "iteration", "document_id", "relevance"]
)

# Basic checks
print("Total relevance rows:", len(qrels_df))
print("\nRelevance label distribution:")
print(qrels_df["relevance"].value_counts().sort_index())

# Relevant cases per query
positive_counts = (
    qrels_df[qrels_df["relevance"] == 1]
    .groupby("query_id")["document_id"]
    .nunique()
)

print("\nRelevant cases per query:")
print(positive_counts.to_string())

# Check duplicate query-document pairs
duplicates = qrels_df.duplicated(
    subset=["query_id", "document_id"]
).sum()

print("\nDuplicate query-document pairs:", duplicates)

# Check training and evaluation query counts
train_queries = [f"AILA_Q{i}" for i in range(1, 11)]
test_queries = [f"AILA_Q{i}" for i in range(11, 51)]

print("\nTraining queries:", len(train_queries))
print("Evaluation queries:", len(test_queries))


Total relevance rows: 145700

Relevance label distribution:
relevance
0    145505
1       195
Name: count, dtype: int64

Relevant cases per query:
query_id
AILA_Q1      2
AILA_Q10     3
AILA_Q11     2
AILA_Q12     1
AILA_Q13     3
AILA_Q14     1
AILA_Q15     1
AILA_Q16     2
AILA_Q17     3
AILA_Q18     1
AILA_Q19     4
AILA_Q2      2
AILA_Q20     2
AILA_Q21     4
AILA_Q22     2
AILA_Q23     2
AILA_Q24     1
AILA_Q25     6
AILA_Q26     8
AILA_Q27     5
AILA_Q28     2
AILA_Q29    22
AILA_Q3      1
AILA_Q30    10
AILA_Q31     2
AILA_Q32     3
AILA_Q33     1
AILA_Q34    13
AILA_Q35     2
AILA_Q36     6
AILA_Q37     3
AILA_Q38     5
AILA_Q39    12
AILA_Q4      1
AILA_Q40     2
AILA_Q41     5
AILA_Q42     4
AILA_Q43     3
AILA_Q44     6
AILA_Q45     2
AILA_Q46     9
AILA_Q47     1
AILA_Q48     4
AILA_Q49     4
AILA_Q5      5
AILA_Q50     2
AILA_Q6      4
AILA_Q7      1
AILA_Q8      3
AILA_Q9      2

Duplicate query-document pairs: 0

Training queries: 10
Evaluation queries: 40


In [ ]:

import os
import re
from rank_bm25 import BM25Okapi

# 1. Locate the case documents
case_folder = os.path.join(dataset_path, "Object_casedocs")

# 2. Read all case documents
case_ids = []
case_texts = []

for filename in sorted(os.listdir(case_folder)):
    if re.fullmatch(r"C\d+\.txt", filename):
        file_path = os.path.join(case_folder, filename)

        with open(file_path, "r", encoding="utf-8", errors="replace") as f:
            text = f.read()

        case_ids.append(filename.replace(".txt", ""))
        case_texts.append(text)

print("Case documents loaded:", len(case_texts))

# 3. Create a simple tokenizer
def tokenize(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

# 4. Convert every case into tokens
tokenized_cases = [tokenize(text) for text in case_texts]

print("Tokenization completed!")

# 5. Build the BM25 index
bm25 = BM25Okapi(tokenized_cases)

print("BM25 index created successfully!")
print("Total indexed cases:", len(case_ids))

# 6. Inspect one document
print("\nExample document ID:", case_ids[0])
print("Number of tokens:", len(tokenized_cases[0]))


Case documents loaded: 2914
Tokenization completed!
BM25 index created successfully!
Total indexed cases: 2914

Example document ID: C1
Number of tokens: 1583


In [ ]:

# 1. Load the first legal query
query_path = os.path.join(dataset_path, "Query_doc.txt")

with open(query_path, "r", encoding="utf-8") as f:
    first_query_line = f.readline().strip()

query_id, query_text = first_query_line.split("||", 1)

# 2. Tokenize the query
query_tokens = tokenize(query_text)

# 3. Calculate BM25 scores for all cases
scores = bm25.get_scores(query_tokens)

# 4. Get the top 5 ranked cases
import numpy as np

top_indices = np.argsort(scores)[::-1][:5]

print("Query ID:", query_id)
print("\nTOP 5 RETRIEVED LEGAL CASES\n")

for rank, idx in enumerate(top_indices, start=1):
    print(f"Rank: {rank}")
    print("Case ID:", case_ids[idx])
    print("BM25 Score:", round(float(scores[idx]), 4))
    print("Case Text:", case_texts[idx][:300].replace("\n", " "))
    print("-" * 60)


Query ID: AILA_Q1

TOP 5 RETRIEVED LEGAL CASES

Rank: 1
Case ID: C1651
BM25 Score: 2579.6886
Case Text: S.B.I. and others v Palak Modi and another Supreme Court of India  3 December 2012 C.A. Nos. 7841-43 of 2012 The Judgment was delivered by : G. S. Singhvi, J. 1. Whether the alleged use of unfair means by Palak Modi and Prabhat Dixit (hereinafter described as 'the private respondents') in the test h
------------------------------------------------------------
Rank: 2
Case ID: C174
BM25 Score: 2523.0298
Case Text: Manager, R.B.I., Bangalore v S. Mani and Others Supreme Court of India  14 March 2005 Appeal (Civil)  6306-6316 of 2003 The Judgment was delivered by : S. B. Sinha, J. 1.  The Respondents herein were Ticca Mazdoors working under the Appellant herein. Ticca Mazdoors are intermittently appointed by th
------------------------------------------------------------
Rank: 3
Case ID: C2517
BM25 Score: 2516.9104
Case Text: Chairman-Cm/w-M.D., T.N.C.S. Corporation Limited and Others v

In [ ]:

# 1. Get the actual relevant cases for AILA_Q1
relevant_cases = set(
    qrels_df[
        (qrels_df["query_id"] == "AILA_Q1") &
        (qrels_df["relevance"] == 1)
    ]["document_id"]
)

# 2. Get the retrieved top 5 case IDs
retrieved_cases = [case_ids[idx] for idx in top_indices]

# 3. Calculate evaluation metrics
precision, recall, f1 = metrics.precision_recall_f1_at_k(
    retrieved_cases, relevant_cases, 5
)

mrr = metrics.reciprocal_rank(
    retrieved_cases, relevant_cases
)

ap = metrics.average_precision(
    retrieved_cases, relevant_cases
)

ndcg = metrics.ndcg_at_k(
    retrieved_cases, relevant_cases, 5
)

# 4. Display results
print("===== AILA_Q1 EVALUATION =====")

print("\nActual relevant cases:", sorted(relevant_cases))
print("Retrieved top 5:", retrieved_cases)

print("\n===== METRICS =====")
print("Precision@5:", round(precision, 4))
print("Recall@5:", round(recall, 4))
print("F1@5:", round(f1, 4))
print("MRR:", round(mrr, 4))
print("AP:", round(ap, 4))
print("nDCG@5:", round(ndcg, 4))

print("\n===== CASE-BY-CASE COMPARISON =====")

for rank, case_id in enumerate(retrieved_cases, start=1):
    status = "RELEVANT" if case_id in relevant_cases else "NOT RELEVANT"
    print(f"Rank {rank}: {case_id} - {status}")


===== AILA_Q1 EVALUATION =====

Actual relevant cases: ['C14', 'C9']
Retrieved top 5: ['C1651', 'C174', 'C2517', 'C1389', 'C1240']

===== METRICS =====
Precision@5: 0.0
Recall@5: 0.0
F1@5: 0.0
MRR: 0.0
AP: 0.0
nDCG@5: 0.0

===== CASE-BY-CASE COMPARISON =====
Rank 1: C1651 - NOT RELEVANT
Rank 2: C174 - NOT RELEVANT
Rank 3: C2517 - NOT RELEVANT
Rank 4: C1389 - NOT RELEVANT
Rank 5: C1240 - NOT RELEVANT


In [ ]:

# Rank all 2,914 cases for AILA_Q1

ranked_indices = np.argsort(scores)[::-1]

# Create a mapping of case ID to ranking position
case_ranks = {
    case_ids[idx]: rank
    for rank, idx in enumerate(ranked_indices, start=1)
}

print("===== RELEVANT CASE RANKS =====")

for case_id in sorted(relevant_cases):
    print(
        case_id,
        "→ Rank:",
        case_ranks.get(case_id, "Not found")
    )

print("\n===== TOP 20 BM25 RESULTS =====")

for rank, idx in enumerate(ranked_indices[:20], start=1):
    case_id = case_ids[idx]
    status = "RELEVANT" if case_id in relevant_cases else "Not relevant"

    print(
        f"Rank {rank}: {case_id} | "
        f"Score: {scores[idx]:.2f} | {status}"
    )


===== RELEVANT CASE RANKS =====
C14 → Rank: 47
C9 → Rank: 1398

===== TOP 20 BM25 RESULTS =====
Rank 1: C1651 | Score: 2579.69 | Not relevant
Rank 2: C174 | Score: 2523.03 | Not relevant
Rank 3: C2517 | Score: 2516.91 | Not relevant
Rank 4: C1389 | Score: 2514.94 | Not relevant
Rank 5: C1240 | Score: 2512.33 | Not relevant
Rank 6: C13 | Score: 2503.79 | Not relevant
Rank 7: C316 | Score: 2496.29 | Not relevant
Rank 8: C560 | Score: 2488.84 | Not relevant
Rank 9: C1837 | Score: 2477.37 | Not relevant
Rank 10: C1483 | Score: 2469.92 | Not relevant
Rank 11: C162 | Score: 2464.78 | Not relevant
Rank 12: C1908 | Score: 2464.52 | Not relevant
Rank 13: C2544 | Score: 2458.63 | Not relevant
Rank 14: C1234 | Score: 2445.90 | Not relevant
Rank 15: C2642 | Score: 2445.70 | Not relevant
Rank 16: C867 | Score: 2445.43 | Not relevant
Rank 17: C128 | Score: 2439.89 | Not relevant
Rank 18: C1020 | Score: 2437.29 | Not relevant
Rank 19: C2638 | Score: 2434.26 | Not relevant
Rank 20: C352 | Score: 2432.

In [ ]:

# Create a mapping between case IDs and their full text
case_text_by_id = dict(zip(case_ids, case_texts))

print("===== RELEVANT CASE C14 =====\n")
print(case_text_by_id["C14"][:1200])

print("\n" + "=" * 70)

print("\n===== RELEVANT CASE C9 =====\n")
print(case_text_by_id["C9"][:1200])

print("\n" + "=" * 70)

print("\n===== TOP RANKED CASE C1651 =====\n")
print(case_text_by_id["C1651"][:1200])


===== RELEVANT CASE C14 =====

Central Inland Water Transport Corporation Limited and Another v Brojo Nath Ganguly and Another
Supreme Court of India

6 April 1986
C.A. No. 4412 and 4413 of 1985
The Judgment was delivered by : D. P. Madon, J.
1. These Appeals by Special Leave granted by this Court raise two questions of considerable importance to Government companies and their employees including their officers. These questions are:
1) Whether a Government company as defined in s. 617 of the Companies Act, 1956, is "the State" within the meaning of Art. 12 of the Constitution?
2) Whether an unconscionable term in a contract of employment is void u/s. 23 of the Indian Contract Act, 1872, as being opposed to public policy and, when such a term is contained in a contract of employment entered into with a Government company, is also void as infringing Art. 14 of the Constitution in case a Government company is "the State" under Art. 12 of the Constitution?
2. Although the record of these A

In [ ]:

import numpy as np
import pandas as pd

# Load all 50 queries
query_path = os.path.join(dataset_path, "Query_doc.txt")

query_dict = {}

with open(query_path, "r", encoding="utf-8") as f:
    for line in f:
        if "||" in line:
            qid, text = line.strip().split("||", 1)
            query_dict[qid] = text

# Evaluate only the 10 development queries
development_ids = [f"AILA_Q{i}" for i in range(1, 11)]

results = []

for qid in development_ids:

    # Get query text
    query_text = query_dict[qid]

    # Retrieve all cases
    query_tokens = tokenize(query_text)
    scores = bm25.get_scores(query_tokens)

    # Rank the top 5
    ranked_indices = np.argsort(scores)[::-1]
    top_indices = ranked_indices[:5]

    retrieved = [case_ids[idx] for idx in top_indices]

    # Get relevant cases
    relevant = set(
        qrels_df[
            (qrels_df["query_id"] == qid) &
            (qrels_df["relevance"] == 1)
        ]["document_id"]
    )

    # Calculate metrics
    precision, recall, f1 = metrics.precision_recall_f1_at_k(
        retrieved, relevant, 5
    )

    mrr = metrics.reciprocal_rank(retrieved, relevant)
    ap = metrics.average_precision(retrieved, relevant)
    ndcg = metrics.ndcg_at_k(retrieved, relevant, 5)

    results.append({
        "Query": qid,
        "Relevant Cases": len(relevant),
        "Precision@5": precision,
        "Recall@5": recall,
        "F1@5": f1,
        "MRR": mrr,
        "AP": ap,
        "nDCG@5": ndcg
    })

# Display individual query results
results_df = pd.DataFrame(results)

print("===== BM25 DEVELOPMENT RESULTS =====")
display(results_df.round(4))

# Calculate average performance
print("\n===== AVERAGE BM25 PERFORMANCE =====")
display(
    results_df.drop(columns=["Query", "Relevant Cases"])
    .mean()
    .to_frame("Average Score")
    .round(4)
)


===== BM25 DEVELOPMENT RESULTS =====


,Query,Relevant Cases,Precision@5,Recall@5,F1@5,MRR,AP,nDCG@5
0,AILA_Q1,2,0.0,0.0,0.0,0.0,0.0,0.0
1,AILA_Q2,2,0.0,0.0,0.0,0.0,0.0,0.0
2,AILA_Q3,1,0.0,0.0,0.0,0.0,0.0,0.0
3,AILA_Q4,1,0.0,0.0,0.0,0.0,0.0,0.0
4,AILA_Q5,5,0.0,0.0,0.0,0.0,0.0,0.0
5,AILA_Q6,4,0.0,0.0,0.0,0.0,0.0,0.0
6,AILA_Q7,1,0.0,0.0,0.0,0.0,0.0,0.0
7,AILA_Q8,3,0.0,0.0,0.0,0.0,0.0,0.0
8,AILA_Q9,2,0.0,0.0,0.0,0.0,0.0,0.0
9,AILA_Q10,3,0.0,0.0,0.0,0.0,0.0,0.0



===== AVERAGE BM25 PERFORMANCE =====


,Average Score
Precision@5,0.0
Recall@5,0.0
F1@5,0.0
MRR,0.0
AP,0.0
nDCG@5,0.0


In [ ]:

print("===== BM25 RANKING DIAGNOSTIC =====\n")

for qid in development_ids:

    # Get the query
    query_text = query_dict[qid]
    query_tokens = tokenize(query_text)

    # Rank all cases
    query_scores = bm25.get_scores(query_tokens)
    ranked_indices = np.argsort(query_scores)[::-1]

    ranked_case_ids = [case_ids[i] for i in ranked_indices]

    # Get actual relevant cases
    relevant = set(
        qrels_df[
            (qrels_df["query_id"] == qid) &
            (qrels_df["relevance"] == 1)
        ]["document_id"].astype(str)
    )

    # Find the positions of relevant cases
    relevant_ranks = [
        ranked_case_ids.index(case_id) + 1
        for case_id in relevant
        if case_id in ranked_case_ids
    ]

    relevant_ranks.sort()

    print(qid)
    print("Relevant cases:", len(relevant))
    print("Relevant case ranks:", relevant_ranks)
    print("Hits in top 5:", sum(r <= 5 for r in relevant_ranks))
    print("Hits in top 20:", sum(r <= 20 for r in relevant_ranks))
    print("Hits in top 100:", sum(r <= 100 for r in relevant_ranks))
    print("-" * 45)


===== BM25 RANKING DIAGNOSTIC =====

AILA_Q1
Relevant cases: 2
Relevant case ranks: [47, 1398]
Hits in top 5: 0
Hits in top 20: 0
Hits in top 100: 1
---------------------------------------------
AILA_Q2
Relevant cases: 2
Relevant case ranks: [275, 572]
Hits in top 5: 0
Hits in top 20: 0
Hits in top 100: 0
---------------------------------------------
AILA_Q3
Relevant cases: 1
Relevant case ranks: [1508]
Hits in top 5: 0
Hits in top 20: 0
Hits in top 100: 0
---------------------------------------------
AILA_Q4
Relevant cases: 1
Relevant case ranks: [949]
Hits in top 5: 0
Hits in top 20: 0
Hits in top 100: 0
---------------------------------------------
AILA_Q5
Relevant cases: 5
Relevant case ranks: [179, 441, 682, 745, 777]
Hits in top 5: 0
Hits in top 20: 0
Hits in top 100: 0
---------------------------------------------
AILA_Q6
Relevant cases: 4
Relevant case ranks: [158, 299, 535, 599]
Hits in top 5: 0
Hits in top 20: 0
Hits in top 100: 0
---------------------------------------------

In [ ]:

from sentence_transformers import SentenceTransformer
import numpy as np

# Load our semantic embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Semantic model loaded successfully!")

# Function to divide long legal judgments into smaller chunks
def split_into_chunks(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

# Divide all case documents into chunks
chunk_texts = []
chunk_case_ids = []

for case_id, text in zip(case_ids, case_texts):

    chunks = split_into_chunks(text)

    for chunk in chunks:
        chunk_texts.append(chunk)
        chunk_case_ids.append(case_id)

print("\n===== CHUNKING RESULTS =====")
print("Original cases:", len(case_ids))
print("Total chunks:", len(chunk_texts))
print("First chunk's case ID:", chunk_case_ids[0])
print("First chunk preview:", chunk_texts[0][:300])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic model loaded successfully!

===== CHUNKING RESULTS =====
Original cases: 2914
Total chunks: 41433
First chunk's case ID: C1
First chunk preview: Masud Khan v State Of Uttar Pradesh Supreme Court of India 26 September 1973 Writ Petition No. 117 of 1973 The Judgment was delivered by : A. Alagiriswami, J. 1. Petitioner Masud Khan prays for his release on the ground that he, an Indian citizen has been illegally arrested and confined to, jail und


In [ ]:

print("===== MODEL TOKEN LIMIT CHECK =====")

print("Maximum model sequence length:", model.max_seq_length)

# Check the first 10 chunks
for i in range(10):
    token_count = len(
        model.tokenizer(
            chunk_texts[i],
            truncation=False
        )["input_ids"]
    )

    print(
        f"Chunk {i+1}: {token_count} tokens | "
        f"Case: {chunk_case_ids[i]}"
    )


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (398 > 256). Running this sequence through the model will result in indexing errors


===== MODEL TOKEN LIMIT CHECK =====
Maximum model sequence length: 256
Chunk 1: 398 tokens | Case: C1
Chunk 2: 366 tokens | Case: C1
Chunk 3: 363 tokens | Case: C1
Chunk 4: 380 tokens | Case: C1
Chunk 5: 384 tokens | Case: C1
Chunk 6: 359 tokens | Case: C1
Chunk 7: 52 tokens | Case: C1
Chunk 8: 445 tokens | Case: C10
Chunk 9: 372 tokens | Case: C10
Chunk 10: 376 tokens | Case: C10


In [ ]:

# Rebuild chunks using the model's tokenizer

chunk_texts = []
chunk_case_ids = []

CHUNK_SIZE = 200
OVERLAP = 40

tokenizer = model.tokenizer

for case_id, text in zip(case_ids, case_texts):

    # Convert the judgment into tokens
    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    start = 0

    while start < len(token_ids):

        end = start + CHUNK_SIZE
        chunk_ids = token_ids[start:end]

        # Convert tokens back into readable text
        chunk = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        if chunk.strip():
            chunk_texts.append(chunk)
            chunk_case_ids.append(case_id)

        start += CHUNK_SIZE - OVERLAP

print("===== UPDATED CHUNKING RESULTS =====")
print("Original cases:", len(case_ids))
print("Total chunks:", len(chunk_texts))
print("First chunk case ID:", chunk_case_ids[0])


===== UPDATED CHUNKING RESULTS =====
Original cases: 2914
Total chunks: 80931
First chunk case ID: C1


In [ ]:

print("===== TOKEN LIMIT VERIFICATION =====")

token_counts = []

for chunk in chunk_texts[:100]:
    count = len(
        tokenizer(
            chunk,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )

    token_counts.append(count)

print("Model maximum:", model.max_seq_length)
print("Maximum tokens in first 100 chunks:", max(token_counts))
print("Minimum tokens:", min(token_counts))

if max(token_counts) <= model.max_seq_length:
    print("\nSUCCESS: All checked chunks fit the model!")
else:
    print("\nWARNING: Some chunks exceed the model limit.")


===== TOKEN LIMIT VERIFICATION =====
Model maximum: 256
Maximum tokens in first 100 chunks: 205
Minimum tokens: 22

SUCCESS: All checked chunks fit the model!


In [ ]:

print("Checking all chunks...")

all_token_counts = []

for i, chunk in enumerate(chunk_texts):

    count = len(
        tokenizer(
            chunk,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )

    all_token_counts.append(count)

print("\n===== COMPLETE TOKEN VALIDATION =====")
print("Total chunks:", len(all_token_counts))
print("Maximum tokens:", max(all_token_counts))
print("Minimum tokens:", min(all_token_counts))
print("Chunks exceeding model limit:",
      sum(x > model.max_seq_length for x in all_token_counts))

assert max(all_token_counts) <= model.max_seq_length

print("\nSUCCESS: Entire collection is ready for embedding!")


Checking all chunks...

===== COMPLETE TOKEN VALIDATION =====
Total chunks: 80931
Maximum tokens: 206
Minimum tokens: 3
Chunks exceeding model limit: 0

SUCCESS: Entire collection is ready for embedding!


In [ ]:

import os
import numpy as np
import pandas as pd

# Create the folder for processed data
processed_path = "/content/drive/MyDrive/legal-retrieval-eval/data/processed"
os.makedirs(processed_path, exist_ok=True)

print("Generating embeddings for all legal chunks...")
print("Total chunks:", len(chunk_texts))
print("Batch size: 32")

# Generate normalized embeddings
embeddings = model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\n===== EMBEDDING RESULTS =====")
print("Embedding shape:", embeddings.shape)
print("Embedding data type:", embeddings.dtype)

# Save embeddings
embedding_path = os.path.join(
    processed_path, "aila_semantic_embeddings.npy"
)

np.save(embedding_path, embeddings)

# Save the mapping between chunks and original case IDs
metadata = pd.DataFrame({
    "chunk_index": range(len(chunk_case_ids)),
    "case_id": chunk_case_ids
})

metadata_path = os.path.join(
    processed_path, "aila_chunk_metadata.csv"
)

metadata.to_csv(metadata_path, index=False)

print("\n===== FILES SAVED =====")
print("Embeddings:", embedding_path)
print("Metadata:", metadata_path)
print("Saved chunks:", len(metadata))


Generating embeddings for all legal chunks...
Total chunks: 80931
Batch size: 32


Batches:   0%|          | 0/2530 [00:00<?, ?it/s]


===== EMBEDDING RESULTS =====
Embedding shape: (80931, 384)
Embedding data type: float32

===== FILES SAVED =====
Embeddings: /content/drive/MyDrive/legal-retrieval-eval/data/processed/aila_semantic_embeddings.npy
Metadata: /content/drive/MyDrive/legal-retrieval-eval/data/processed/aila_chunk_metadata.csv
Saved chunks: 80931


In [ ]:

import os
import numpy as np
import pandas as pd

processed_path = "/content/drive/MyDrive/legal-retrieval-eval/data/processed"

embedding_path = os.path.join(
    processed_path, "aila_semantic_embeddings.npy"
)

metadata_path = os.path.join(
    processed_path, "aila_chunk_metadata.csv"
)

print("===== SAVED FILE VERIFICATION =====")

print("Embeddings file exists:", os.path.exists(embedding_path))
print("Metadata file exists:", os.path.exists(metadata_path))

# Load existing files instead of regenerating embeddings
embeddings = np.load(embedding_path)
metadata = pd.read_csv(metadata_path)

print("\nEmbedding shape:", embeddings.shape)
print("Metadata shape:", metadata.shape)
print("First five case IDs:")
print(metadata["case_id"].head().tolist())


===== SAVED FILE VERIFICATION =====
Embeddings file exists: True
Metadata file exists: True

Embedding shape: (80931, 384)
Metadata shape: (80931, 2)
First five case IDs:
['C1', 'C1', 'C1', 'C1', 'C1']


In [ ]:

# Retrieve cases for AILA_Q1

query_id = "AILA_Q1"
query_text = query_dict[query_id]

# Generate query embedding
query_embedding = model.encode(
    query_text,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Calculate cosine similarity with all chunks
chunk_scores = embeddings @ query_embedding

# Connect each chunk score to its original case
score_df = pd.DataFrame({
    "case_id": metadata["case_id"],
    "similarity": chunk_scores
})

# Keep the highest chunk similarity for each case
case_scores = (
    score_df.groupby("case_id")["similarity"]
    .max()
    .sort_values(ascending=False)
)

# Retrieve top 10 cases
top_cases = case_scores.head(10)

print("===== SEMANTIC RETRIEVAL: AILA_Q1 =====")

for rank, (case_id, score) in enumerate(top_cases.items(), 1):
    print(f"Rank: {rank} | Case: {case_id} | Score: {score:.4f}")

# Check the actual relevant cases
relevant_cases = set(
    qrels_df[
        (qrels_df["query_id"] == query_id) &
        (qrels_df["relevance"] == 1)
    ]["document_id"].astype(str)
)

print("\n===== RELEVANT CASE CHECK =====")

for case_id in relevant_cases:
    if case_id in case_scores.index:
        rank = case_scores.index.get_loc(case_id) + 1
        print(f"{case_id} → Rank: {rank}")


===== SEMANTIC RETRIEVAL: AILA_Q1 =====
Rank: 1 | Case: C1613 | Score: 0.6355
Rank: 2 | Case: C1240 | Score: 0.6115
Rank: 3 | Case: C560 | Score: 0.6053
Rank: 4 | Case: C190 | Score: 0.6019
Rank: 5 | Case: C306 | Score: 0.5971
Rank: 6 | Case: C1389 | Score: 0.5821
Rank: 7 | Case: C1405 | Score: 0.5800
Rank: 8 | Case: C2611 | Score: 0.5795
Rank: 9 | Case: C2328 | Score: 0.5762
Rank: 10 | Case: C38 | Score: 0.5741

===== RELEVANT CASE CHECK =====
C9 → Rank: 1150
C14 → Rank: 30


In [ ]:

semantic_results = []

for qid in development_ids:

    print("Evaluating:", qid)

    # Get query text
    query_text = query_dict[qid]

    # Generate query embedding
    query_embedding = model.encode(
        query_text,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Calculate similarity with all chunks
    chunk_scores = embeddings @ query_embedding

    # Map chunk scores to original cases
    score_df = pd.DataFrame({
        "case_id": metadata["case_id"],
        "similarity": chunk_scores
    })

    # Use maximum chunk similarity for each case
    case_scores = (
        score_df.groupby("case_id")["similarity"]
        .max()
        .sort_values(ascending=False)
    )

    # Retrieve top 5 cases
    retrieved = case_scores.head(5).index.tolist()

    # Get relevant cases
    relevant = set(
        qrels_df[
            (qrels_df["query_id"] == qid) &
            (qrels_df["relevance"] == 1)
        ]["document_id"].astype(str)
    )

    # Calculate metrics using the same functions as BM25
    precision, recall, f1 = metrics.precision_recall_f1_at_k(
        retrieved, relevant, 5
    )

    mrr = metrics.reciprocal_rank(retrieved, relevant)
    ap = metrics.average_precision(retrieved, relevant)
    ndcg = metrics.ndcg_at_k(retrieved, relevant, 5)

    semantic_results.append({
        "Query": qid,
        "Relevant Cases": len(relevant),
        "Precision@5": precision,
        "Recall@5": recall,
        "F1@5": f1,
        "MRR": mrr,
        "AP": ap,
        "nDCG@5": ndcg
    })

# Display individual results
semantic_df = pd.DataFrame(semantic_results)

print("\n===== SEMANTIC RETRIEVAL RESULTS =====")
display(semantic_df.round(4))

# Calculate average performance
print("\n===== AVERAGE SEMANTIC PERFORMANCE =====")

semantic_average = (
    semantic_df.drop(columns=["Query", "Relevant Cases"])
    .mean()
)

display(semantic_average.to_frame("Average Score").round(4))


Evaluating: AILA_Q1
Evaluating: AILA_Q2
Evaluating: AILA_Q3
Evaluating: AILA_Q4
Evaluating: AILA_Q5
Evaluating: AILA_Q6
Evaluating: AILA_Q7
Evaluating: AILA_Q8
Evaluating: AILA_Q9
Evaluating: AILA_Q10

===== SEMANTIC RETRIEVAL RESULTS =====


,Query,Relevant Cases,Precision@5,Recall@5,F1@5,MRR,AP,nDCG@5
0,AILA_Q1,2,0.0,0.0,0.0,0.0,0.0,0.0
1,AILA_Q2,2,0.0,0.0,0.0,0.0,0.0,0.0
2,AILA_Q3,1,0.0,0.0,0.0,0.0,0.0,0.0
3,AILA_Q4,1,0.0,0.0,0.0,0.0,0.0,0.0
4,AILA_Q5,5,0.0,0.0,0.0,0.0,0.0,0.0
5,AILA_Q6,4,0.0,0.0,0.0,0.0,0.0,0.0
6,AILA_Q7,1,0.0,0.0,0.0,0.0,0.0,0.0
7,AILA_Q8,3,0.0,0.0,0.0,0.0,0.0,0.0
8,AILA_Q9,2,0.0,0.0,0.0,0.0,0.0,0.0
9,AILA_Q10,3,0.0,0.0,0.0,0.0,0.0,0.0



===== AVERAGE SEMANTIC PERFORMANCE =====


,Average Score
Precision@5,0.0
Recall@5,0.0
F1@5,0.0
MRR,0.0
AP,0.0
nDCG@5,0.0


In [ ]:

import numpy as np
import pandas as pd

corrected_results = []
rank_diagnostics = []

for qid in development_ids:

    query_text = query_dict[qid]

    # Generate query embedding
    query_embedding = model.encode(
        query_text,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Calculate similarity for every chunk
    chunk_scores = embeddings @ query_embedding

    # Aggregate chunk scores into case scores
    score_df = pd.DataFrame({
        "case_id": metadata["case_id"],
        "similarity": chunk_scores
    })

    case_scores = (
        score_df.groupby("case_id")["similarity"]
        .max()
        .sort_values(ascending=False)
    )

    # Full ranked list
    ranked_cases = case_scores.index.tolist()

    # Relevant cases for this query
    relevant = set(
        qrels_df[
            (qrels_df["query_id"] == qid) &
            (qrels_df["relevance"] == 1)
        ]["document_id"].astype(str)
    )

    # Find ranks of all relevant cases
    relevant_ranks = [
        rank for rank, case_id in enumerate(ranked_cases, 1)
        if case_id in relevant
    ]

    top5 = ranked_cases[:5]
    hits = len(set(top5) & relevant)

    # Top-5 metrics
    precision = hits / 5
    recall = hits / len(relevant) if relevant else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0
    )

    # Full-ranking MRR
    mrr = 1 / relevant_ranks[0] if relevant_ranks else 0

    # Full-ranking Average Precision
    precision_at_relevant = [
        sum(1 for x in ranked_cases[:rank] if x in relevant) / rank
        for rank in relevant_ranks
    ]

    ap = (
        sum(precision_at_relevant) / len(relevant)
        if relevant else 0
    )

    # Binary nDCG@5
    dcg = sum(
        1 / np.log2(rank + 1)
        for rank, case_id in enumerate(top5, 1)
        if case_id in relevant
    )

    ideal_hits = min(len(relevant), 5)
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    ndcg = dcg / idcg if idcg else 0

    corrected_results.append({
        "Query": qid,
        "Precision@5": precision,
        "Recall@5": recall,
        "F1@5": f1,
        "MRR": mrr,
        "AP": ap,
        "nDCG@5": ndcg
    })

    rank_diagnostics.append({
        "Query": qid,
        "Relevant Cases": len(relevant),
        "Relevant Ranks": relevant_ranks,
        "Hits@5": hits
    })

corrected_df = pd.DataFrame(corrected_results)
diagnostics_df = pd.DataFrame(rank_diagnostics)

print("===== CORRECTED SEMANTIC METRICS =====")
display(corrected_df.round(4))

print("\n===== AVERAGE PERFORMANCE =====")
display(
    corrected_df.drop(columns="Query")
    .mean()
    .to_frame("Average Score")
    .round(4)
)

print("\n===== RELEVANT CASE RANKS =====")
display(diagnostics_df)


===== CORRECTED SEMANTIC METRICS =====


,Query,Precision@5,Recall@5,F1@5,MRR,AP,nDCG@5
0,AILA_Q1,0.0,0.0,0,0.0333,0.0175,0.0
1,AILA_Q2,0.0,0.0,0,0.0066,0.0044,0.0
2,AILA_Q3,0.0,0.0,0,0.0005,0.0005,0.0
3,AILA_Q4,0.0,0.0,0,0.0010,0.0010,0.0
4,AILA_Q5,0.0,0.0,0,0.0013,0.0025,0.0
5,AILA_Q6,0.0,0.0,0,0.0132,0.0116,0.0
6,AILA_Q7,0.0,0.0,0,0.0050,0.0050,0.0
7,AILA_Q8,0.0,0.0,0,0.0025,0.0015,0.0
8,AILA_Q9,0.0,0.0,0,0.0033,0.0027,0.0
9,AILA_Q10,0.0,0.0,0,0.0035,0.0030,0.0



===== AVERAGE PERFORMANCE =====


,Average Score
Precision@5,0.000
Recall@5,0.000
F1@5,0.000
MRR,0.007
AP,0.005
nDCG@5,0.000



===== RELEVANT CASE RANKS =====


,Query,Relevant Cases,Relevant Ranks,Hits@5
0,AILA_Q1,2,"[30, 1150]",0
1,AILA_Q2,2,"[152, 939]",0
2,AILA_Q3,1,[2199],0
3,AILA_Q4,1,[981],0
4,AILA_Q5,5,"[799, 916, 932, 1375, 1577]",0
5,AILA_Q6,4,"[76, 91, 506, 748]",0
6,AILA_Q7,1,[202],0
7,AILA_Q8,3,"[403, 2494, 2567]",0
8,AILA_Q9,2,"[301, 1010]",0
9,AILA_Q10,3,"[287, 605, 1401]",0


In [ ]:

print("===== QUERY TOKEN LENGTH DIAGNOSTIC =====")

for qid in development_ids:

    query_text = query_dict[qid]

    token_count = len(
        tokenizer(
            query_text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )

    status = (
        "EXCEEDS LIMIT"
        if token_count > model.max_seq_length
        else "WITHIN LIMIT"
    )

    print(
        f"{qid}: {token_count} tokens | {status}"
    )

print("\nModel maximum:", model.max_seq_length)


===== QUERY TOKEN LENGTH DIAGNOSTIC =====
AILA_Q1: 1200 tokens | EXCEEDS LIMIT
AILA_Q2: 602 tokens | EXCEEDS LIMIT
AILA_Q3: 236 tokens | WITHIN LIMIT
AILA_Q4: 499 tokens | EXCEEDS LIMIT
AILA_Q5: 959 tokens | EXCEEDS LIMIT
AILA_Q6: 540 tokens | EXCEEDS LIMIT
AILA_Q7: 293 tokens | EXCEEDS LIMIT
AILA_Q8: 737 tokens | EXCEEDS LIMIT
AILA_Q9: 796 tokens | EXCEEDS LIMIT
AILA_Q10: 436 tokens | EXCEEDS LIMIT

Model maximum: 256
